In [1]:
from pathlib import Path
from PIL import Image
import shutil, os, json, subprocess

# Copy project code
CODE_SRC = Path("/kaggle/input/datasets/dmmehedihasanabid/icmla26-updated-code/icmla26_code")
CODE_DST = Path("/kaggle/working/icmla26_code")

if CODE_DST.exists():
    shutil.rmtree(CODE_DST)

shutil.copytree(CODE_SRC, CODE_DST)
os.chdir(CODE_DST)

subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
subprocess.run("python check_environment.py", shell=True, check=True)

# Prepare clean standardized dataset folders
PREP = Path("/kaggle/working/icmla_prepared")
if PREP.exists():
    shutil.rmtree(PREP)
PREP.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def safe_link_tree(src_dir, dst_dir, verify=False, prefix=""):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    copied = 0
    bad = 0

    for i, p in enumerate(sorted(src_dir.glob("*"))):
        if not p.is_file() or p.suffix.lower() not in IMG_EXTS:
            continue

        if verify:
            try:
                with Image.open(p) as img:
                    img.verify()
            except Exception:
                bad += 1
                continue

        safe_prefix = f"{prefix}_" if prefix else ""
        out = dst_dir / f"{safe_prefix}{i:06d}_{p.name}"

        os.symlink(p, out)
        copied += 1

    return copied, bad

# 1. PlantVillage source
PV_ROOT = Path("/kaggle/input/datasets/arjuntejaswi/plant-village/PlantVillage")

print("PlantVillage")
print("Early", safe_link_tree(PV_ROOT / "Potato___Early_blight", PREP / "plantvillage_potato/Early Blight", prefix="pv_early"))
print("Late", safe_link_tree(PV_ROOT / "Potato___Late_blight", PREP / "plantvillage_potato/Late Blight", prefix="pv_late"))
print("Healthy", safe_link_tree(PV_ROOT / "Potato___healthy", PREP / "plantvillage_potato/Healthy", prefix="pv_healthy"))

# 2. PLD Pakistan target
PLD_ROOT = Path("/kaggle/input/datasets/rizwan123456789/potato-disease-leaf-datasetpld/PLD_3_Classes_256")

print("PLD Pakistan")
for split in ["Training", "Validation", "Testing"]:
    print(split, "Early", safe_link_tree(PLD_ROOT / split / "Early_Blight", PREP / "pld_pakistan/Early Blight", verify=True, prefix=f"pld_{split}_early"))
    print(split, "Late", safe_link_tree(PLD_ROOT / split / "Late_Blight", PREP / "pld_pakistan/Late Blight", verify=True, prefix=f"pld_{split}_late"))
    print(split, "Healthy", safe_link_tree(PLD_ROOT / split / "Healthy", PREP / "pld_pakistan/Healthy", verify=True, prefix=f"pld_{split}_healthy"))

# 3. Irish target
IRISH_ROOT = Path("/kaggle/input/datasets/dmmehedihasanabid/irish-potato/Irish for Kaggle")

print("Irish")
for src_cls, dst_cls, pref in [
    ("Early blight", "Early Blight", "irish_early"),
    ("Late blight", "Late Blight", "irish_late"),
    ("Healthy", "Healthy", "irish_healthy")
]:
    copied, bad = safe_link_tree(IRISH_ROOT / src_cls, PREP / f"irish_potato/{dst_cls}", verify=True, prefix=pref)
    print(src_cls, "copied:", copied, "bad:", bad)

# 4. PlantDoc potato-only overlap
PLANTDOC_ROOT = Path("/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset")

print("PlantDoc potato overlap")
for split in ["train", "test"]:
    print(split, "Early", safe_link_tree(PLANTDOC_ROOT / split / "Potato leaf early blight", PREP / "plantdoc_potato_overlap/Early Blight", verify=True, prefix=f"plantdoc_{split}_early"))
    print(split, "Late", safe_link_tree(PLANTDOC_ROOT / split / "Potato leaf late blight", PREP / "plantdoc_potato_overlap/Late Blight", verify=True, prefix=f"plantdoc_{split}_late"))

# Canonical config
cfg = {
    "inherits": "configs/default_config.json",
    "output_dir": "/kaggle/working/paper_outputs",
    "datasets": {
        "source": {
            "name": "PlantVillage Potato",
            "domain": "laboratory",
            "path": "/kaggle/working/icmla_prepared/plantvillage_potato"
        },
        "targets": [
            {
                "name": "PLD Pakistan",
                "domain": "regional_field_cropped",
                "path": "/kaggle/working/icmla_prepared/pld_pakistan"
            },
            {
                "name": "Irish Potato Dataset",
                "domain": "field",
                "path": "/kaggle/working/icmla_prepared/irish_potato",
                "fixed_split_per_class": {
                    "train": 998,
                    "val": 200,
                    "test": 300,
                    "seed": 2026
                }
            },
            {
                "name": "PlantDoc Potato Overlap",
                "domain": "uncontrolled_field_partial_overlap",
                "path": "/kaggle/working/icmla_prepared/plantdoc_potato_overlap",
                "optional": True
            }
        ]
    }
}

Path("configs/kaggle_config.json").write_text(json.dumps(cfg, indent=2))

print("Setup complete.")
print(Path("configs/kaggle_config.json").read_text())

# Final count check
print("\nFinal prepared counts:")
for d in sorted(PREP.rglob("*")):
    if d.is_dir() and d.parent != PREP and len(list(d.glob("*"))) > 0:
        print(len(list(d.glob("*"))), d)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OK required numpy: 2.0.2
OK required pandas: 2.3.3
OK required scipy: 1.16.3
OK required sklearn: 1.6.1
OK required PIL: 11.3.0
OK required matplotlib: 3.10.0
OK required seaborn: 0.13.2
OK required torch: 2.10.0+cu128
OK required torchvision: 0.25.0+cu128
OK optional cv2: 4.13.0
OK optional timm: 1.0.26
OK optional open_clip: 3.3.0
OK optional transformers: 5.0.0
OK optional umap: 0.5.12
PlantVillage
Early (1000, 0)
Late (1000, 0)
Healthy (152, 0)
PLD Pakistan
Training Early (1303, 0)
Training Late (1132, 0)
Training Healthy (816, 0)
Validation Early (163, 0)
Validation Late (151, 0)
Validation Healthy (102, 0)
Testing Early (162, 0)
Testing Late (141, 0)
Testing Healthy (102, 0)
Irish
Early blight copied: 1498 bad: 2
Late blight copied: 1500 bad: 0
Healthy copied: 1500 bad:

In [2]:
from pathlib import Path
import json, subprocess

STAGE_NAME = "stage2_resnet"
MODELS = ["resnet50"]
BATCH_SIZE = 32
OUTPUT_DIR = f"/kaggle/working/paper_outputs_{STAGE_NAME}"

stage_cfg = {
    "inherits": "configs/kaggle_config.json",
    "output_dir": OUTPUT_DIR,
    "models": MODELS,
    "batch_size": BATCH_SIZE
}

Path(f"configs/{STAGE_NAME}_config.json").write_text(json.dumps(stage_cfg, indent=2))

subprocess.run(f"rm -rf {OUTPUT_DIR}", shell=True, check=False)

subprocess.run(
    f"python run_pipeline.py --config configs/{STAGE_NAME}_config.json --skip-foundation",
    shell=True,
    check=True
)

subprocess.run(
    f'zip -r /kaggle/working/{STAGE_NAME}_outputs.zip "{OUTPUT_DIR}" > /dev/null',
    shell=True,
    check=True
)

print(f"Download: /kaggle/working/{STAGE_NAME}_outputs.zip")

Using device: cuda
Class mapping:
  0: Early Blight
  1: Late Blight
  2: Healthy

PlantVillage Potato: 2152 images
  Early Blight: 1000
  Late Blight: 1000
  Healthy: 152

PLD Pakistan: 4072 images
  Early Blight: 1628
  Late Blight: 1424
  Healthy: 1020

Irish Potato Dataset: 4498 images
  Early Blight: 1498
  Late Blight: 1500
  Healthy: 1500

PlantDoc Potato Overlap: 222 images
  Early Blight: 117
  Late Blight: 105
  Healthy: 0

Training resnet50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 165MB/s] 
evidence resnet50: 100%|██████████| 64/64 [00:06<00:00,  9.26it/s]
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/kaggle/working/icmla26_code/src/pipeline.py:612: RuntimeWarning: invalid value encountered in cast
  arr[mask] = np.median(arr[~mask], axis=0)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/kaggle/working/icmla26_code/src/pipeline.py:612: RuntimeWarning: invalid value encountered in cast
  arr[mask] = np.median(


Done. Outputs saved to /kaggle/working/paper_outputs_stage2_resnet
Download: /kaggle/working/stage2_resnet_outputs.zip


In [ ]:
from pathlib import Path
from IPython.display import display, Markdown, FileLink, HTML, Javascript
import subprocess, os

STAGE_NAME = "stage2_resnet"
OUTPUT_DIR = Path(f"/kaggle/working/paper_outputs_{STAGE_NAME}")
ZIP_PATH = Path(f"/kaggle/working/{STAGE_NAME}_outputs.zip")

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Output folder not found: {OUTPUT_DIR}")

# Recreate zip safely
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

subprocess.run(
    f'zip -r "{ZIP_PATH}" "{OUTPUT_DIR}" > /dev/null',
    shell=True,
    check=True
)

os.chdir("/kaggle/working")

display(Markdown(f"""
# Download Ready

Created:

`{ZIP_PATH}`

Size:

`{ZIP_PATH.stat().st_size / (1024**2):.2f} MB`

Download from the Kaggle right-side **Output** panel, or click below.
"""))

display(FileLink(ZIP_PATH.name))

display(HTML(f"""
<a id="download_link" href="{ZIP_PATH.name}" download="{ZIP_PATH.name}"
   style="font-size:20px; font-weight:bold; color:white; background:#2563eb;
          padding:12px 18px; border-radius:8px; text-decoration:none; display:inline-block;">
Download {ZIP_PATH.name}
</a>
"""))

display(Javascript("""
setTimeout(() => {
    const a = document.getElementById("download_link");
    if (a) a.click();
}, 1000);
"""))